<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Mercosul/nic_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importações e Configurações

In [ ]:
import requests
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.io as pio
import re
import time

pio.renderers.default = "notebook_connected"

BASE_URL = "https://www.nic.br"
HEADERS = {"User-Agent": "Mozilla/5.0"}

DATABASE_NAME = "internet_governance_news.db"

print("✅ Ambiente NIC.br pronto")


In [ ]:
def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela prontos")

create_database()


In [ ]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()


## Scraper

In [ ]:
def carregar_indice_json(codigo="todos", arquivo=1):
    url = f"{BASE_URL}/noticias/indice/{codigo}/indice-{arquivo}.json"
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.json()


In [ ]:
def scrape_nic_noticias(total_noticias=1150):
    print("🔄 Scraping NIC.br via JSON...")
    start = time.time()
    dados = carregar_indice_json("todos", 1)

    artigos = []

    for noticia in dados[:total_noticias]:
        title = noticia.get("titulo")
        url = noticia.get("permalink")
        date = noticia.get("data")
        resumo = noticia.get("resumo")

        data = {
            "title": title,
            "date": date,
            "author": "NIC.br",
            "url": url,
            "source": "NIC Notícias"
        }

        artigos.append(data)
        insert_article(**data)

    df = pd.DataFrame(artigos)
    print(f"✅ {len(df)} notícias coletadas")
    print(f"⏱️ {time.time() - start:.2f}s")
    display(df.head(10))
    return df

df_nic = scrape_nic_noticias(total_noticias=1150)


In [ ]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT title, date, author, url, source
        FROM articles
        WHERE source = 'NIC Notícias'
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total NIC no banco: {len(df_db)}")
display(df_db.head(20))


## Filtro + Gráficos

In [ ]:
keywords = [
    "internet", "governança", "dados",
    "segurança", "privacidade", "digital", "IA"
]

pattern = "|".join(keywords)

df_filt = df_db[
    df_db["title"].str.contains(pattern, case=False, na=False)
]

print(f"🔎 Filtradas: {len(df_filt)}")


In [ ]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados")
        return

    # --------------------------
    # Top 15 Notícias
    # --------------------------
    top15 = df.head(15).copy()
    top15["rank"] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x="rank",
        y="title",
        orientation="h",
        title="Top 15 Notícias – NIC.br"
    )
    fig1.update_layout(
        height=600,
        yaxis={"categoryorder": "array",
               "categoryarray": top15["title"][::-1]}
    )
    fig1.show()

    # --------------------------
    # Distribuição por Fonte
    # --------------------------
    src = df["source"].value_counts().reset_index()
    src.columns = ["source", "count"]

    fig2 = px.pie(
        src,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # --------------------------
    # Nuvem de Palavras
    # --------------------------
    text = " ".join(df["title"].astype(str)).lower()
    words = re.findall(r"\b[a-záàâãéèêíïóôõöúçñ]{4,}\b", text)

    wc = (
        pd.Series(words)
        .value_counts()
        .head(25)
        .reset_index()
    )
    wc.columns = ["palavra", "freq"]

    fig3 = px.treemap(
        wc,
        path=["palavra"],
        values="freq",
        title="Nuvem de Palavras – NIC.br"
    )
    fig3.show()


In [ ]:
# %%
plot_charts(df_filt if not df_filt.empty else df_db)
